In [ ]:
#@title Fase 5: Reporte Final de Métricas y Recomendación
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from scipy.optimize import fsolve

# --- 1. CONFIGURACIÓN Y DATOS ---
precio_objetivo = 12000 #@param {type:"number"}
n_prueba = 48  # Tamaño del Hold-out (últimas 48 horas)

# Separación de datos
entrenamiento = df_analisis.iloc[:-n_prueba] # Todo menos las últimas 48h
prueba = df_analisis.iloc[-n_prueba:]        # Solo las últimas 48h (Naranja)

# --- 2. CÁLCULOS INTERNOS (Matemática oculta) ---
# Modelo para validar (entrenado con el pasado)
coef_val = np.polyfit(entrenamiento['t'], entrenamiento['close_price'], 3)
modelo_val = np.poly1d(coef_val)

# Modelo para proyectar (entrenado con todo)
coef_futuro = np.polyfit(df_analisis['t'], df_analisis['close_price'], 3)
modelo_futuro = np.poly1d(coef_futuro)

# Cálculo de errores
y_real = prueba['close_price'].values
y_pred_val = modelo_val(prueba['t'].values) # Predicción interna para calcular el error

rmse_val = np.sqrt(mean_squared_error(y_real, y_pred_val))
precio_medio = y_real.mean()
error_relativo = (rmse_val / precio_medio) * 100
distancia_usd = precio_objetivo - df_analisis['close_price'].iloc[-1]

# --- 3. GRÁFICA ---
plt.figure(figsize=(12, 6))

# Historial (Azul)
plt.plot(entrenamiento['t'], entrenamiento['close_price'],
         color='navy', alpha=0.4, label='Historial de Entrenamiento')

# Realidad Oculta / Hold-out (Naranja)
plt.plot(prueba['t'], y_real,
         color='orange', linewidth=2.5, label='Realidad (Hold-out)')

# Proyección Futura (Roja - La línea que pediste mantener)
t_final = df_analisis['t'].max()
t_proy = np.linspace(t_final, t_final + (n_prueba * 1.5), 50) # Proyecta 72h a futuro
plt.plot(t_proy, modelo_futuro(t_proy),
         color='red', linestyle='--', linewidth=2, label='Proyección Futura')

# Línea de Objetivo
plt.axhline(y=precio_objetivo, color='black', linestyle=':', label=f'Objetivo: ${precio_objetivo}')

plt.title('Fase 5: Validación Hold-Out y Proyección')
plt.xlabel('Tiempo (t)')
plt.ylabel('Precio (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- 4. REPORTE IMPRESO (Formato solicitado) ---
print("--- REPORTE FINAL DE CALIDAD ---")
print(f"Error Absoluto (RMSE): ${rmse_val:,.2f} USD")
print(f"ERROR RELATIVO: {error_relativo:.2f}%")
print(f"ANÁLISIS DE PRECISIÓN: {'Alta' if error_relativo < 5 else 'Aceptable' if error_relativo < 15 else 'Baja'}")
print("-" * 50)
print(f"Precio objetivo de prueba colocado: ${precio_objetivo:,.2f} USD")
print(f"Diferencia actual respecto al objetivo: ${distancia_usd:,.2f} USD")
print("-" * 50)

# Lógica para la recomendación final
# Verifica si la proyección futura se acerca al objetivo
precio_futuro_estimado = modelo_futuro(t_final + 24)
va_hacia_objetivo = False

if distancia_usd > 0: # El objetivo es subir
    if precio_futuro_estimado > df_analisis['close_price'].iloc[-1]:
        va_hacia_objetivo = True
else: # El objetivo es bajar (short)
    if precio_futuro_estimado < df_analisis['close_price'].iloc[-1]:
        va_hacia_objetivo = True

if error_relativo < 15 and va_hacia_objetivo:
    print("RECOMENDACIÓN FINAL: RECOMENDADO: El objetivo es coherente con la inercia del modelo.")
else:
    print("RECOMENDACIÓN FINAL: NO RECOMENDADO: El objetivo es divergente o la fiabilidad es insuficiente.")